## Import Library

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import json

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

## Mengatur Lokasi File

In [2]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 120)

direktori_aktif = Path.cwd().resolve()
nama_file_dataset = "PhiUSIIL_Phishing_URL_Dataset.csv"

daftar_kandidat_lokasi = [
    direktori_aktif / nama_file_dataset,
    direktori_aktif / "data" / "raw" / nama_file_dataset,
    direktori_aktif.parent / nama_file_dataset,
    direktori_aktif.parent / "data" / "raw" / nama_file_dataset,
    direktori_aktif.parent.parent / nama_file_dataset,
    direktori_aktif.parent.parent / "data" / "raw" / nama_file_dataset,
]

lokasi_dataset = None

for lokasi in daftar_kandidat_lokasi:
    if lokasi.exists():
        lokasi_dataset = lokasi
        break

if lokasi_dataset is None:
    hasil_pencarian = list(direktori_aktif.rglob(nama_file_dataset))

    if len(hasil_pencarian) > 0:
        lokasi_dataset = hasil_pencarian[0]
    else:
        raise FileNotFoundError("Dataset tidak ditemukan. Pastikan file CSV berada di folder utama project atau data/raw.")

direktori_project = lokasi_dataset.parent.parent.parent if lokasi_dataset.parent.name == "raw" else lokasi_dataset.parent

direktori_data_processed = direktori_project / "data" / "processed"
direktori_output = direktori_project / "reports" / "outputs"

direktori_data_processed.mkdir(parents=True, exist_ok=True)
direktori_output.mkdir(parents=True, exist_ok=True)

print("Direktori aktif notebook:", direktori_aktif)
print("Lokasi dataset:", lokasi_dataset)
print("Direktori project:", direktori_project)
print("Folder data processed:", direktori_data_processed)
print("Folder output:", direktori_output)

Direktori aktif notebook: C:\Users\ASUS\PHISHING\notebooks
Lokasi dataset: C:\Users\ASUS\PHISHING\data\raw\PhiUSIIL_Phishing_URL_Dataset.csv
Direktori project: C:\Users\ASUS\PHISHING
Folder data processed: C:\Users\ASUS\PHISHING\data\processed
Folder output: C:\Users\ASUS\PHISHING\reports\outputs


## Membaca Dataset

In [3]:
data_asli = pd.read_csv(lokasi_dataset)

print("Dataset berhasil dibaca.")
print("Jumlah baris:", data_asli.shape[0])
print("Jumlah kolom:", data_asli.shape[1])

data_asli.head()

Dataset berhasil dibaca.
Jumlah baris: 235795
Jumlah kolom: 56


,FILENAME,URL,URLLength,Domain,DomainLength,IsDomainIP,TLD,URLSimilarityIndex,CharContinuationRate,TLDLegitimateProb,URLCharProb,TLDLength,NoOfSubDomain,HasObfuscation,NoOfObfuscatedChar,ObfuscationRatio,NoOfLettersInURL,LetterRatioInURL,NoOfDegitsInURL,DegitRatioInURL,NoOfEqualsInURL,NoOfQMarkInURL,NoOfAmpersandInURL,NoOfOtherSpecialCharsInURL,SpacialCharRatioInURL,IsHTTPS,LineOfCode,LargestLineLength,HasTitle,Title,DomainTitleMatchScore,URLTitleMatchScore,HasFavicon,Robots,IsResponsive,NoOfURLRedirect,NoOfSelfRedirect,HasDescription,NoOfPopup,NoOfiFrame,HasExternalFormSubmit,HasSocialNet,HasSubmitButton,HasHiddenFields,HasPasswordField,Bank,Pay,Crypto,HasCopyrightInfo,NoOfImage,NoOfCSS,NoOfJS,NoOfSelfRef,NoOfEmptyRef,NoOfExternalRef,label
0,521848.txt,https://www.southbankmosaics.com,31,www.southbankmosaics.com,24,0,com,100.0,1.000000,0.522907,0.061933,3,1,0,0,0.0,18,0.581,0,0.0,0,0,0,1,0.032,1,558,9381,1,à¸‚à¹ˆà¸²à¸§à¸ªà¸” à¸‚à¹ˆà¸²à¸§à¸§à¸±à¸™à¸™à¸µ...,0.000000,0.000000,0,1,1,0,0,0,0,1,0,0,1,1,0,1,0,0,1,34,20,28,119,0,124,1
1,31372.txt,https://www.uni-mainz.de,23,www.uni-mainz.de,16,0,de,100.0,0.666667,0.032650,0.050207,2,1,0,0,0.0,9,0.391,0,0.0,0,0,0,2,0.087,1,618,9381,1,johannes gutenberg-universitÃ¤t mainz,55.555556,55.555556,1,1,0,0,0,0,0,0,0,1,1,0,0,0,0,0,1,50,9,8,39,0,217,1
2,597387.txt,https://www.voicefmradio.co.uk,29,www.voicefmradio.co.uk,22,0,uk,100.0,0.866667,0.028555,0.064129,2,2,0,0,0.0,15,0.517,0,0.0,0,0,0,2,0.069,1,467,682,1,voice fm southampton,46.666667,46.666667,0,1,1,0,0,1,0,0,0,0,1,1,0,0,0,0,1,10,2,7,42,2,5,1
3,554095.txt,https://www.sfnmjournal.com,26,www.sfnmjournal.com,19,0,com,100.0,1.000000,0.522907,0.057606,3,1,0,0,0.0,13,0.500,0,0.0,0,0,0,1,0.038,1,6356,26824,1,home page: seminars in fetal and neonatal medi...,0.000000,0.000000,0,1,1,0,0,0,1,12,0,1,1,1,0,0,1,1,1,3,27,15,22,1,31,1
4,151578.txt,https://www.rewildingargentina.org,33,www.rewildingargentina.org,26,0,org,100.0,1.000000,0.079963,0.059441,3,1,0,0,0.0,20,0.606,0,0.0,0,0,0,1,0.030,1,6089,28404,1,fundaciÃ³n rewilding argentina,100.000000,100.000000,0,1,1,1,1,1,0,2,0,1,1,1,0,1,1,0,1,244,15,34,72,1,85,1


## Validasi Dasar Dataset

In [4]:
nama_kolom_target_asli = "label"

if nama_kolom_target_asli not in data_asli.columns:
    raise ValueError("Kolom target 'label' tidak ditemukan.")

jumlah_missing = data_asli.isna().sum().sum()
jumlah_duplikasi = data_asli.duplicated().sum()

print("Jumlah missing value:", jumlah_missing)
print("Jumlah data duplikat:", jumlah_duplikasi)
print("Nilai unik label asli:", sorted(data_asli[nama_kolom_target_asli].unique()))

Jumlah missing value: 0
Jumlah data duplikat: 0
Nilai unik label asli: [np.int64(0), np.int64(1)]


## Membuat Target Baru

In [5]:
data_pra_proses = data_asli.copy()

data_pra_proses["target_phishing"] = data_pra_proses["label"].map({
    0: 1,
    1: 0
})

data_pra_proses["keterangan_target"] = data_pra_proses["target_phishing"].map({
    1: "Phishing",
    0: "Legitimate"
})

ringkasan_target = data_pra_proses["keterangan_target"].value_counts().reset_index()
ringkasan_target.columns = ["kategori", "jumlah_data"]

ringkasan_target

,kategori,jumlah_data
0,Legitimate,134850
1,Phishing,100945


## Menentukan Kolom yang Tidak Dipakai Langsung

In [6]:
kolom_tidak_dipakai_langsung = [
    "FILENAME",
    "URL",
    "Domain",
    "Title",
    "label",
    "keterangan_target"
]

kolom_tidak_dipakai_langsung = [
    kolom for kolom in kolom_tidak_dipakai_langsung
    if kolom in data_pra_proses.columns
]

print("Kolom yang tidak dipakai langsung:")
print(kolom_tidak_dipakai_langsung)

Kolom yang tidak dipakai langsung:
['FILENAME', 'URL', 'Domain', 'Title', 'label', 'keterangan_target']


## Menyiapkan Data Referensi

In [7]:
kolom_referensi = [
    "FILENAME",
    "URL",
    "Domain",
    "TLD",
    "label",
    "target_phishing",
    "keterangan_target"
]

kolom_referensi = [
    kolom for kolom in kolom_referensi
    if kolom in data_pra_proses.columns
]

data_referensi = data_pra_proses[kolom_referensi].copy()

data_referensi.head()

,FILENAME,URL,Domain,TLD,label,target_phishing,keterangan_target
0,521848.txt,https://www.southbankmosaics.com,www.southbankmosaics.com,com,1,0,Legitimate
1,31372.txt,https://www.uni-mainz.de,www.uni-mainz.de,de,1,0,Legitimate
2,597387.txt,https://www.voicefmradio.co.uk,www.voicefmradio.co.uk,uk,1,0,Legitimate
3,554095.txt,https://www.sfnmjournal.com,www.sfnmjournal.com,com,1,0,Legitimate
4,151578.txt,https://www.rewildingargentina.org,www.rewildingargentina.org,org,1,0,Legitimate


## Membersihkan Kolom TLD

In [8]:
data_pra_proses["TLD"] = data_pra_proses["TLD"].astype(str).str.lower().str.strip()
data_pra_proses["TLD"] = data_pra_proses["TLD"].replace(["nan", "none", ""], "tidak_diketahui")

ringkasan_tld = data_pra_proses["TLD"].value_counts().head(20).reset_index()
ringkasan_tld.columns = ["tld", "jumlah_data"]

ringkasan_tld

,tld,jumlah_data
0,com,112554
1,org,18793
2,net,7097
3,app,6508
4,uk,6395
5,co,5422
6,io,4201
7,de,3996
8,ru,3875
9,au,2979


## Membagi Data Train dan Test

In [9]:
target = data_pra_proses["target_phishing"]

data_fitur_awal = data_pra_proses.drop(columns=kolom_tidak_dipakai_langsung, errors="ignore")
data_fitur_awal = data_fitur_awal.drop(columns=["target_phishing"], errors="ignore")

X_train_awal, X_test_awal, y_train, y_test = train_test_split(
    data_fitur_awal,
    target,
    test_size=0.2,
    random_state=42,
    stratify=target
)

print("Jumlah data train:", X_train_awal.shape[0])
print("Jumlah data test:", X_test_awal.shape[0])
print("Jumlah fitur awal:", X_train_awal.shape[1])

print("\nDistribusi target train:")
print(y_train.value_counts(normalize=True).round(4))

print("\nDistribusi target test:")
print(y_test.value_counts(normalize=True).round(4))

Jumlah data train: 188636
Jumlah data test: 47159
Jumlah fitur awal: 51

Distribusi target train:
target_phishing
0    0.5719
1    0.4281
Name: proportion, dtype: float64

Distribusi target test:
target_phishing
0    0.5719
1    0.4281
Name: proportion, dtype: float64


## Mengolah TLD Berdasarkan Data Train

In [10]:
jumlah_top_tld = 20

top_tld_train = X_train_awal["TLD"].value_counts().head(jumlah_top_tld).index.tolist()

def ringkas_tld(nilai_tld, daftar_top_tld):
    if nilai_tld in daftar_top_tld:
        return nilai_tld
    return "lainnya"

X_train_awal["TLD_ringkas"] = X_train_awal["TLD"].apply(lambda nilai: ringkas_tld(nilai, top_tld_train))
X_test_awal["TLD_ringkas"] = X_test_awal["TLD"].apply(lambda nilai: ringkas_tld(nilai, top_tld_train))

print("Top TLD dari data train:")
print(top_tld_train)

Top TLD dari data train:
['com', 'org', 'net', 'app', 'uk', 'co', 'io', 'de', 'ru', 'au', 'dev', 'top', 'jp', 'it', 'br', 'fr', 'edu', 'nl', 'ca', 'info']


## One-Hot Encoding TLD 

In [11]:
X_train_tld = pd.get_dummies(X_train_awal["TLD_ringkas"], prefix="TLD", dtype=int)
X_test_tld = pd.get_dummies(X_test_awal["TLD_ringkas"], prefix="TLD", dtype=int)

X_test_tld = X_test_tld.reindex(columns=X_train_tld.columns, fill_value=0)

print("Jumlah fitur TLD hasil encoding:", X_train_tld.shape[1])
X_train_tld.head()

Jumlah fitur TLD hasil encoding: 21


,TLD_app,TLD_au,TLD_br,TLD_ca,TLD_co,TLD_com,TLD_de,TLD_dev,TLD_edu,TLD_fr,TLD_info,TLD_io,TLD_it,TLD_jp,TLD_lainnya,TLD_net,TLD_nl,TLD_org,TLD_ru,TLD_top,TLD_uk
76039,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
145208,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0
30237,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
81469,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
104833,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0


## Menyiapkan Fitur Numerik

In [12]:
kolom_numerik = X_train_awal.select_dtypes(include=["int64", "float64"]).columns.tolist()

print("Jumlah kolom numerik:", len(kolom_numerik))
print(kolom_numerik)

Jumlah kolom numerik: 50
['URLLength', 'DomainLength', 'IsDomainIP', 'URLSimilarityIndex', 'CharContinuationRate', 'TLDLegitimateProb', 'URLCharProb', 'TLDLength', 'NoOfSubDomain', 'HasObfuscation', 'NoOfObfuscatedChar', 'ObfuscationRatio', 'NoOfLettersInURL', 'LetterRatioInURL', 'NoOfDegitsInURL', 'DegitRatioInURL', 'NoOfEqualsInURL', 'NoOfQMarkInURL', 'NoOfAmpersandInURL', 'NoOfOtherSpecialCharsInURL', 'SpacialCharRatioInURL', 'IsHTTPS', 'LineOfCode', 'LargestLineLength', 'HasTitle', 'DomainTitleMatchScore', 'URLTitleMatchScore', 'HasFavicon', 'Robots', 'IsResponsive', 'NoOfURLRedirect', 'NoOfSelfRedirect', 'HasDescription', 'NoOfPopup', 'NoOfiFrame', 'HasExternalFormSubmit', 'HasSocialNet', 'HasSubmitButton', 'HasHiddenFields', 'HasPasswordField', 'Bank', 'Pay', 'Crypto', 'HasCopyrightInfo', 'NoOfImage', 'NoOfCSS', 'NoOfJS', 'NoOfSelfRef', 'NoOfEmptyRef', 'NoOfExternalRef']


## Menggabungkan Fitur Numerik dan TLD

In [13]:
X_train_model = pd.concat(
    [
        X_train_awal[kolom_numerik].reset_index(drop=True),
        X_train_tld.reset_index(drop=True)
    ],
    axis=1
)

X_test_model = pd.concat(
    [
        X_test_awal[kolom_numerik].reset_index(drop=True),
        X_test_tld.reset_index(drop=True)
    ],
    axis=1
)

y_train_model = y_train.reset_index(drop=True)
y_test_model = y_test.reset_index(drop=True)

print("Ukuran X_train_model:", X_train_model.shape)
print("Ukuran X_test_model:", X_test_model.shape)
print("Ukuran y_train_model:", y_train_model.shape)
print("Ukuran y_test_model:", y_test_model.shape)

Ukuran X_train_model: (188636, 71)
Ukuran X_test_model: (47159, 71)
Ukuran y_train_model: (188636,)
Ukuran y_test_model: (47159,)


## Mengecek Missing Value Setelah Pra-Proses

In [14]:
jumlah_missing_train = X_train_model.isna().sum().sum()
jumlah_missing_test = X_test_model.isna().sum().sum()

print("Missing value X_train:", jumlah_missing_train)
print("Missing value X_test:", jumlah_missing_test)

Missing value X_train: 0
Missing value X_test: 0


## Mengecek Konsistensi Kolom Train dan Test

In [15]:
kolom_train = list(X_train_model.columns)
kolom_test = list(X_test_model.columns)

if kolom_train != kolom_test:
    raise ValueError("Kolom train dan test tidak sama.")

print("Kolom train dan test sudah konsisten.")
print("Jumlah fitur akhir:", len(kolom_train))

Kolom train dan test sudah konsisten.
Jumlah fitur akhir: 71


## Scaling untuk Model Linear

In [16]:
scaler = StandardScaler()

X_train_scaled_array = scaler.fit_transform(X_train_model)
X_test_scaled_array = scaler.transform(X_test_model)

X_train_scaled = pd.DataFrame(
    X_train_scaled_array,
    columns=X_train_model.columns
)

X_test_scaled = pd.DataFrame(
    X_test_scaled_array,
    columns=X_test_model.columns
)

print("Scaling selesai.")
print("Ukuran X_train_scaled:", X_train_scaled.shape)
print("Ukuran X_test_scaled:", X_test_scaled.shape)

Scaling selesai.
Ukuran X_train_scaled: (188636, 71)
Ukuran X_test_scaled: (47159, 71)


## Menyimpan Data Hasil Pra-Proses

In [17]:
lokasi_X_train = direktori_data_processed / "X_train.csv"
lokasi_X_test = direktori_data_processed / "X_test.csv"
lokasi_y_train = direktori_data_processed / "y_train.csv"
lokasi_y_test = direktori_data_processed / "y_test.csv"

lokasi_X_train_scaled = direktori_data_processed / "X_train_scaled.csv"
lokasi_X_test_scaled = direktori_data_processed / "X_test_scaled.csv"

lokasi_data_referensi = direktori_data_processed / "data_referensi.csv"

X_train_model.to_csv(lokasi_X_train, index=False)
X_test_model.to_csv(lokasi_X_test, index=False)
y_train_model.to_csv(lokasi_y_train, index=False, header=["target_phishing"])
y_test_model.to_csv(lokasi_y_test, index=False, header=["target_phishing"])

X_train_scaled.to_csv(lokasi_X_train_scaled, index=False)
X_test_scaled.to_csv(lokasi_X_test_scaled, index=False)

data_referensi.to_csv(lokasi_data_referensi, index=False)

print("Data hasil pra-proses berhasil disimpan.")
print("Folder:", direktori_data_processed)

Data hasil pra-proses berhasil disimpan.
Folder: C:\Users\ASUS\PHISHING\data\processed


## Menyimpan Daftar Fitur dan Metadata

In [18]:
daftar_fitur_model = pd.DataFrame({
    "nomor": range(1, len(X_train_model.columns) + 1),
    "nama_fitur": X_train_model.columns
})

lokasi_daftar_fitur = direktori_output / "daftar_fitur_model.csv"
daftar_fitur_model.to_csv(lokasi_daftar_fitur, index=False)

metadata_pra_proses = {
    "jumlah_data_awal": int(data_asli.shape[0]),
    "jumlah_kolom_awal": int(data_asli.shape[1]),
    "jumlah_data_train": int(X_train_model.shape[0]),
    "jumlah_data_test": int(X_test_model.shape[0]),
    "jumlah_fitur_model": int(X_train_model.shape[1]),
    "target_model": "target_phishing",
    "arti_target": {
        "1": "Phishing",
        "0": "Legitimate"
    },
    "test_size": 0.2,
    "random_state": 42,
    "jumlah_top_tld": jumlah_top_tld,
    "top_tld_train": top_tld_train,
    "kolom_tidak_dipakai_langsung": kolom_tidak_dipakai_langsung
}

lokasi_metadata = direktori_output / "metadata_pra_proses.json"

with open(lokasi_metadata, "w", encoding="utf-8") as file:
    json.dump(metadata_pra_proses, file, indent=4, ensure_ascii=False)

print("Daftar fitur disimpan di:", lokasi_daftar_fitur)
print("Metadata pra-proses disimpan di:", lokasi_metadata)

Daftar fitur disimpan di: C:\Users\ASUS\PHISHING\reports\outputs\daftar_fitur_model.csv
Metadata pra-proses disimpan di: C:\Users\ASUS\PHISHING\reports\outputs\metadata_pra_proses.json


## Ringkasan Hasil Pra-Proses

In [19]:
print("RINGKASAN PRA-PROSES DATA")
print("-" * 45)
print("Jumlah data awal:", data_asli.shape[0])
print("Jumlah kolom awal:", data_asli.shape[1])
print("Jumlah data train:", X_train_model.shape[0])
print("Jumlah data test:", X_test_model.shape[0])
print("Jumlah fitur akhir:", X_train_model.shape[1])
print("Target model: target_phishing")
print("Arti target: 1 = Phishing, 0 = Legitimate")

print("\nDistribusi target train:")
print(y_train_model.value_counts().sort_index())

print("\nDistribusi target test:")
print(y_test_model.value_counts().sort_index())

print("\nFile hasil pra-proses:")
print(lokasi_X_train)
print(lokasi_X_test)
print(lokasi_y_train)
print(lokasi_y_test)
print(lokasi_X_train_scaled)
print(lokasi_X_test_scaled)

RINGKASAN PRA-PROSES DATA
---------------------------------------------
Jumlah data awal: 235795
Jumlah kolom awal: 56
Jumlah data train: 188636
Jumlah data test: 47159
Jumlah fitur akhir: 71
Target model: target_phishing
Arti target: 1 = Phishing, 0 = Legitimate

Distribusi target train:
target_phishing
0    107880
1     80756
Name: count, dtype: int64

Distribusi target test:
target_phishing
0    26970
1    20189
Name: count, dtype: int64

File hasil pra-proses:
C:\Users\ASUS\PHISHING\data\processed\X_train.csv
C:\Users\ASUS\PHISHING\data\processed\X_test.csv
C:\Users\ASUS\PHISHING\data\processed\y_train.csv
C:\Users\ASUS\PHISHING\data\processed\y_test.csv
C:\Users\ASUS\PHISHING\data\processed\X_train_scaled.csv
C:\Users\ASUS\PHISHING\data\processed\X_test_scaled.csv
